# CADIP staging and AUXIP staging on-demand flows

Demonstration of flows defined on those two stories:   
https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-715   
https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-718   

## 1 - Initialisation

In [ ]:
# Access to Prefect
import os
print(f"Prefect server URL used internally: {os.environ['PREFECT_API_URL']}")
dashboard = f"{os.environ['RSPY_PREFECT_URL']}/dashboard"
print(f"Prefect dashboard public URL: {dashboard}")

In [ ]:
# Choose prefect deployment method
from resources.widget_utils import *
deploy_prefect_radio

In [ ]:
# Choose prefect flow run method
run_prefect_radio

In [ ]:
# Init environment before running a demo notebook.
from resources.utils import *  
init_demo()
# Reload the global vars again
from resources.utils import *  

from resources.dask_clusters.dask_main_env import *
await init_dask_cluster_staging()

In [ ]:
# Create a test collection
CATALOG_COLLECTION_ID = "SPRINT26_TEST_COLLECTION"
collection = create_test_collection(CATALOG_COLLECTION_ID)

# Check that it is empty
items = catalog_client.get_items(CATALOG_COLLECTION_ID)
assert not list(items)

# Other test values
SESSION_ID = "S1A_20200105072204051312"
CADIP_COLLECTION_ID = "sgs_sentinel1"

In [ ]:
# Other imports
import os
from pystac import ItemCollection
from rs_common.prefect_utils import *

## 2 - Set up flow parameters

In [ ]:
cadip_flow_parameters = {
  "env": {
    "owner_id": OWNER_ID,
  },
  "cadip_collection_identifier": CADIP_COLLECTION_ID,
  "session_identifier": SESSION_ID,
  "catalog_collection_identifier": CATALOG_COLLECTION_ID
}

aux_flow_parameters = {
  "env": {
    "owner_id": OWNER_ID,
  },
  "start_datetime": "2024-05-27T09:44:09.509000Z",
  "end_datetime": "2024-05-27T09:44:19.509000Z",
  "product_type": "AUX_PP2",
  "catalog_collection_identifier": CATALOG_COLLECTION_ID
}

## 3 - Deploy Prefect flows

In [ ]:
# Deploy the Prefect flow
cadip_deploy, aux_deploy = await deploy_prefect(
    deploy_file="./cadip_auxip_staging_flows.yaml", 
    s3_code_folder=f"users/{OWNER_ID}/code", 
    work_pool_name=os.environ["PREFECT_WORK_POOL_INTEGRATED"]
)

## 4 - Run flows

Run one flow for CADIP staging and one for AUXIP staging.

In [ ]:
from rs_workflows.cadip_flow import on_demand_cadip_staging
await run_prefect(
    deploy_name=cadip_deploy, 
    py_func=on_demand_cadip_staging, 
    params=cadip_flow_parameters
)

In [ ]:
from rs_workflows.aux_flow import on_demand_aux_staging
await run_prefect(
    deploy_name=aux_deploy, 
    py_func=on_demand_aux_staging, 
    params=aux_flow_parameters
)

In [ ]:
# Processed items published to the catalog
ItemCollection(list(catalog_client.get_items(CATALOG_COLLECTION_ID)))